Data of 1025 pokemons
dataset of pokemons with description, name of the pokemon from which they evolve

About Dataset
This is a dataset of 1025 pokemons that I scraped from pokeapi.co using python. I made a pokedex in flask using this api ( hamzacyberpatcher.pythonanywhere.com ). This dataset contains the following data:

Pokemon's id
Pokemon's name
Pokemon's rank (whether it is legendary, mythical, baby or an ordinary pokemon)
Pokemon's generation
The pokemon from which this pokemon evolved from
It's primary and secondary type as type1 and type2
It's base stats
It's height (decimetres) and weight (hectograms)
It's abilities (Note: if a pokemon has multiple abilities such as overgrow and chlorophyll than they are separated by a space and if there is a space in the name of the ability it has been replaced by a hyphen for example a pokemon with abilities blaze and solar power it is given as blaze solar-power)
A little description of the pokemon in english (description of pokemons from 1009 is marked as "Not Available").

## Imports

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('pokemons.csv')

In [3]:
display(df.head())#

,id,name,rank,generation,evolves_from,type1,type2,hp,atk,def,spatk,spdef,speed,total,height,weight,abilities,desc
0,1,bulbasaur,ordinary,generation-i,nothing,grass,poison,45,49,49,65,65,45,318,7,69,overgrow chlorophyll,A strange seed was planted on its back at birt...
1,2,ivysaur,ordinary,generation-i,bulbasaur,grass,poison,60,62,63,80,80,60,405,10,130,overgrow chlorophyll,"When the bulb on its back grows large, it appe..."
2,3,venusaur,ordinary,generation-i,ivysaur,grass,poison,80,82,83,100,100,80,525,20,1000,overgrow chlorophyll,The plant blooms when it is absorbing solar en...
3,4,charmander,ordinary,generation-i,nothing,fire,NaN,39,52,43,60,50,65,309,6,85,blaze solar-power,"Obviously prefers hot places. When it rains, s..."
4,5,charmeleon,ordinary,generation-i,charmander,fire,NaN,58,64,58,80,65,80,405,11,190,blaze solar-power,"When it swings its burning tail, it elevates t..."


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 18 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id            1025 non-null   int64 
 1   name          1025 non-null   object
 2   rank          1025 non-null   object
 3   generation    1025 non-null   object
 4   evolves_from  1025 non-null   object
 5   type1         1025 non-null   object
 6   type2         526 non-null    object
 7   hp            1025 non-null   int64 
 8   atk           1025 non-null   int64 
 9   def           1025 non-null   int64 
 10  spatk         1025 non-null   int64 
 11  spdef         1025 non-null   int64 
 12  speed         1025 non-null   int64 
 13  total         1025 non-null   int64 
 14  height        1025 non-null   int64 
 15  weight        1025 non-null   int64 
 16  abilities     1025 non-null   object
 17  desc          1025 non-null   object
dtypes: int64(10), object(8)
memory usage: 144.3+ KB


Dataset tiene 1025 filas (pokemons) como decia en el about y 14 columnas (caracteristicas)

Todas estan completas excepto type2 que tiene 499 valores nulos (pokemons que solo tienen un tipo) Lo confirmamos:

In [7]:
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0])

type2    499
dtype: int64


Rellenamos los nulos de type2 con 'None' para indicar que no tienen segundo tipo

In [8]:
# Rellenar los nulos de type2
df['type2'] = df['type2'].fillna('None')

Es raro porque hay pokemons que no evolucionan de ningun otro pokemon y en la columna evolves_from tienen NaN. Lo miramos:

In [9]:
print(df['evolves_from'].unique()[:10])

['nothing' 'bulbasaur' 'ivysaur' 'charmander' 'charmeleon' 'squirtle'
 'wartortle' 'caterpie' 'metapod' 'weedle']


Como sospechábamos, el dataset no usa valores vacíos (NaN) para los Pokémon que no evolucionan, sino que usa la palabra "nothing". Por eso df.info() te decía que no había nulos, porque para Python esa palabra es un texto tan válido como "Bulbasaur".

In [11]:
print(df['evolves_from'].value_counts().head())

evolves_from
nothing    541
eevee        8
tyrogue      3
applin       3
scyther      2
Name: count, dtype: int64


Opción A (Dejarlo así): Si usas la columna evolves_from tal cual, el modelo tratará 'nothing' como una categoría más. Esto está bien, pero tiene un problema: la columna tendrá cientos de categorías diferentes (una por cada Pokémon que evoluciona), lo cual puede confundir a algunos modelos.

Opción B (Feature Engineering - Recomendada): Crear una nueva columna que simplifique esta información. Al modelo le suele importar más si el Pokémon es una forma base o no, que saber exactamente de quién viene.

In [12]:

# Crear una nueva característica 'is_base'
# 1 = Es base (no evoluciona de nadie)
# 0 = Es evolución (viene de otro)
df['is_base'] = df['evolves_from'].apply(lambda x: 1 if x == 'nothing' else 0)

# Ver cómo queda
display(df[['name', 'evolves_from', 'is_base']].head())

Cantidad de Pokémon base ('nothing'): 541


,name,evolves_from,is_base
0,bulbasaur,nothing,1
1,ivysaur,bulbasaur,0
2,venusaur,ivysaur,0
3,charmander,nothing,1
4,charmeleon,charmander,0


In [13]:
print(df.isnull().sum().sum())

0


In [14]:
duplicados = df.duplicated(subset=['id', 'name']).sum()
print(f"Número de filas duplicadas: {duplicados}")

Número de filas duplicadas: 0


In [15]:
# Revisar valores únicos en columnas categóricas clave
cols_to_check = ['rank', 'generation', 'type1']

for col in cols_to_check:
    print(f"\nValores únicos en '{col}':")
    print(df[col].unique())


Valores únicos en 'rank':
['ordinary' 'legendary' 'mythical' 'baby']

Valores únicos en 'generation':
['generation-i' 'generation-ii' 'generation-iii' 'generation-iv'
 'generation-v' 'generation-vi' 'generation-vii' 'generation-viii'
 'generation-ix']

Valores únicos en 'type1':
['grass' 'fire' 'water' 'bug' 'normal' 'poison' 'electric' 'ground'
 'fairy' 'fighting' 'psychic' 'rock' 'ghost' 'ice' 'dragon' 'dark' 'steel'
 'flying']


In [16]:
# Estadísticas descriptivas para detectar outliers o errores (ej: HP = 0)
display(df.describe())

# Check rápido: ¿Hay algún stat o medida menor o igual a 0?
errores_numericos = df[(df['height'] <= 0) | (df['weight'] <= 0) | (df['hp'] <= 0)]

if not errores_numericos.empty:
    print("¡Alerta! Hay Pokémons con datos numéricos inválidos:")
    display(errores_numericos)
else:
    print("\nTodos los datos numéricos parecen consistentes (mayores a 0).")

,id,hp,atk,def,spatk,spdef,speed,total,height,weight,is_base
count,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000
mean,513.000000,70.184390,77.521951,72.507317,70.080976,70.205854,67.186341,427.686829,12.116098,669.865366,0.527805
std,296.036315,26.631054,29.782541,29.286972,29.658378,26.639329,28.717227,112.770735,12.481673,1212.731138,0.499470
min,1.000000,1.000000,5.000000,5.000000,10.000000,20.000000,5.000000,175.000000,1.000000,1.000000,0.000000
25%,257.000000,50.000000,55.000000,50.000000,47.000000,50.000000,45.000000,325.000000,5.000000,85.000000,0.000000
50%,513.000000,68.000000,75.000000,70.000000,65.000000,67.000000,65.000000,450.000000,10.000000,280.000000,1.000000
75%,769.000000,85.000000,100.000000,90.000000,90.000000,86.000000,88.000000,508.000000,15.000000,700.000000,1.000000
max,1025.000000,255.000000,181.000000,230.000000,173.000000,230.000000,200.000000,720.000000,200.000000,9999.000000,1.000000



Todos los datos numéricos parecen consistentes (mayores a 0).
